OCR avanzado para documentos empresariales
==========================================

En este notebook vamos a ver cómo se utiliza un modelo moderno de OCR/document parsing dentro de un pipeline empresarial. La idea no es solamente "leer texto", sino convertir un PDF o una imagen en una representación útil para procesos posteriores: validación de campos, revisión humana, carga a sistemas transaccionales o RAG.

Modelos recientes como Mistral OCR o DeepSeek-OCR trabajan más cerca de los modelos vision-language: reciben una página como imagen o documento, preservan parte del layout y pueden devolver Markdown, JSON o texto estructurado. Esto resulta útil cuando una factura, un contrato o un formulario contiene tablas, encabezados, sellos o información distribuida visualmente.

## Preparación del ambiente

Instalemos las dependencias necesarias desde el archivo de requerimientos asociado a este notebook. El ejemplo puede ejecutarse sin credenciales usando una respuesta simulada; si dispone de una API key de Mistral, también puede activar la llamada real al servicio.

In [ ]:
!wget -q https://raw.githubusercontent.com/santiagxf/M72109/master/docs/document-understanding/advanced-ocr-enterprise.txt
%pip install -r advanced-ocr-enterprise.txt --quiet

Recordar reiniciar la sesión si se encuentra trabajando en Google Colab y el entorno lo solicita.

## ¿Qué cambió respecto al OCR clásico?

En un pipeline clásico, primero detectamos palabras y luego intentamos reconstruir líneas, tablas y campos con reglas. En un pipeline moderno, el modelo puede producir directamente una salida más rica, por ejemplo Markdown con tablas o JSON con anotaciones. Podemos pensar el resultado como una capa intermedia entre la página visual y las aplicaciones de negocio.

A grandes rasgos, las prácticas que se repiten en sistemas empresariales son:

1. **Normalizar la entrada**: convertir PDF, imagen o escaneo en páginas procesables.
2. **Parsear el documento**: usar OCR avanzado para obtener texto, tablas, figuras y orden de lectura.
3. **Extraer estructura**: transformar el Markdown o texto en un esquema de negocio.
4. **Validar y auditar**: conservar metadatos, confianza, página de origen y evidencia visual.
5. **Integrar**: enviar los campos validados a un ERP, CRM, data lake o índice de búsqueda.

En la investigación previa se revisaron fuentes públicas de Mistral OCR, DeepSeek-OCR, PaddleOCR y MinerU. Todas apuntan a la misma dirección: documentos convertidos a Markdown/JSON, soporte para PDFs e imágenes, manejo de tablas y layouts complejos, y salidas pensadas para LLMs, RAG o flujos de automatización empresarial.

## Documento de ejemplo

Para mantener el notebook autocontenido, crearemos una factura simple como imagen. En un caso real, esta imagen podría venir de un PDF escaneado, de un correo electrónico, de un portal de proveedores o de un bucket de almacenamiento.

In [17]:
!wget -q https://raw.githubusercontent.com/santiagxf/M72109/master/docs/document-understanding/_images/invoice_example.png

![](https://raw.githubusercontent.com/santiagxf/M72109/master/docs/document-understanding/_images/invoice_example.png)

## Opción 1: Mistral OCR como servicio

Mistral OCR expone una API de OCR/document parsing. El patrón típico es subir el archivo, obtener una URL firmada y procesarlo con `mistral-ocr-latest`. La respuesta contiene páginas y Markdown, y opcionalmente imágenes embebidas en base64. Note que no escribimos la API key en el notebook: se lee desde una variable de ambiente.

In [7]:
import json
import os

MISTRAL_API_KEY = "<API-KEY>" # Puede obtener una API de Mistral gratuita en https://console.mistral.ai/

from mistralai.client import Mistral
from mistralai.client.models import ImageURLChunk

client = Mistral(api_key=MISTRAL_API_KEY)

In [20]:
with open("invoice_example.png", "rb") as document:
    uploaded_file = client.files.upload(
        file={
            "file_name": document.name,
            "content": document.read(),
        },
        purpose="ocr",
    )
    signed_url = client.files.get_signed_url(file_id=uploaded_file.id, expiry=1)

In [21]:
respuesta_ocr = client.ocr.process(
    model="mistral-ocr-latest",
    document=ImageURLChunk(image_url=signed_url.url),
    include_image_base64=False,
)
respuesta_ocr = json.loads(respuesta_ocr.model_dump_json())

In [22]:
import json

print(json.dumps(respuesta_ocr, indent=2))

{
  "pages": [
    {
      "index": 0,
      "markdown": "Power Consult IT\nRE: Richard Green\n12345 Testway Rd\nTesting, FL 33318\n\nInvoice #: 8020001544\nInvoice Date: February 02, 2012\n\nPower Consultit\n\nBill To:\nA1 Clock Makers\nJack Marrow\n1234 Watchmakers Way\nNew, NY 10028\n\nINVOICE\n\n|  Description | Quantity | Rate | Total  |\n| --- | --- | --- | --- |\n|  Professional Fees | 14.50 | $100.00 | $1,450.00  |\n|  Fixed Price |  |  | $2,110.00  |\n|  Personal Expenses |  |  | $255.20  |\n|   |  | Total | $3,815.20  |\n\nDetails for Invoice #:8020001544\n\nProfessional Fees\n\n|  Work Date | Work Description | Quantity | Rate | Total  |\n| --- | --- | --- | --- | --- |\n|  05/22/2009 | SoW review and editing. | 1.00 | $ 100.00 | $100.00  |\n|  05/22/2009 | Creatives for infrastructure document. | 4.00 | $ 100.00 | $400.00  |\n|  05/22/2009 | Rendered creatives in various formats and packages for client. | 0.50 | $ 100.00 | $50.00  |\n|  05/26/2009 | Created infrastructure f

El punto importante es que el downstream ya no recibe únicamente una lista de caracteres. Recibe una representación más cercana al documento: títulos, líneas y tablas. Esto simplifica el paso siguiente, que suele ser la extracción de campos de negocio.

## Explorando la salida en Markdown

Veamos la primera página como Markdown. En documentos de muchas páginas, aquí conviene guardar también el número de página, el nombre del archivo, el hash del documento y el timestamp del procesamiento.

In [23]:
from IPython.display import Markdown, display

paginas = respuesta_ocr["pages"]
markdown_pagina = paginas[0]["markdown"]

display(Markdown(markdown_pagina))

Power Consult IT
RE: Richard Green
12345 Testway Rd
Testing, FL 33318

Invoice #: 8020001544
Invoice Date: February 02, 2012

Power Consultit

Bill To:
A1 Clock Makers
Jack Marrow
1234 Watchmakers Way
New, NY 10028

INVOICE

|  Description | Quantity | Rate | Total  |
| --- | --- | --- | --- |
|  Professional Fees | 14.50 | $100.00 | $1,450.00  |
|  Fixed Price |  |  | $2,110.00  |
|  Personal Expenses |  |  | $255.20  |
|   |  | Total | $3,815.20  |

Details for Invoice #:8020001544

Professional Fees

|  Work Date | Work Description | Quantity | Rate | Total  |
| --- | --- | --- | --- | --- |
|  05/22/2009 | SoW review and editing. | 1.00 | $ 100.00 | $100.00  |
|  05/22/2009 | Creatives for infrastructure document. | 4.00 | $ 100.00 | $400.00  |
|  05/22/2009 | Rendered creatives in various formats and packages for client. | 0.50 | $ 100.00 | $50.00  |
|  05/26/2009 | Created infrastructure for Project Alpha and passed on overview sheet to team. | 3.50 | $ 100.00 | $350.00  |
|  05/27/2009 | Created copies of overview and distributed to team. Updated project tracker for client. | 0.50 | $ 100.00 | $50.00  |
|  05/27/2009 | Created a project highlight cheat-sheet, as requested by client, for Richard and Lisa's meeting. | 1.50 | $ 100.00 | $150.00  |
|  05/28/2009 | Met with Lisa to spec out future phase of project. | 2.00 | $ 100.00 | $200.00  |
|  05/29/2009 | Created rough SoW per Lisa's instructions. | 1.50 | $ 100.00 | $150.00  |
|   | Total For Professional Fees | 14.50 |  | $1,450.00  |
|  Fixed Price  |   |   |   |   |
|  Work Date | Work Description |  |  | Total  |
|  05/15/2009 | Fixed-price charge for use of our Analytic software. |  |  | $1,500.00  |
|  05/22/2009 | DVD creation for infrastructure creatives. |  |  | $15.00  |
|  05/24/2009 | Stock photos for Beta project. |  |  | $98.00  |
|  05/24/2009 | binder package of optional stock photos. |  |  | $100.00  |
|  05/28/2009 | stock photos. |  |  | $47.00  |
|  06/01/2009 | Monthly maintenance plan. |  |  | $100.00  |
|  06/01/2009 | Weekly status update charge. |  |  | $100.00  |
|  06/02/2009 | Final Analytics Report. |  |  | $150.00  |
|   | Total For Fixed Price |  |  | $2,110.00  |
|  Personal Expenses  |   |   |   |   |
|  Work Date | Work Description |  |  | Total  |
|  05/24/2009 | Gas and travel charges. |  |  | $23.23  |
|  05/27/2009 | Reimbursable supplies, printouts, etc. |  |  | $58.34  |
|  05/27/2009 | Gas and travel charges. |  |  | $32.99  |
|  06/02/2009 | Reimbursable supplies, printouts, etc. |  |  | $34.16  |
|  06/02/2009 | Installation issue with client's project tracking software. Paid software tech support. Recommended EnterYourHours.com which is web based so it never has installation issues and support is free. |  |  | $50.00  |
|  06/02/2009 | Reimbursable copier and paper charges. |  |  | $22.26  |
|  06/02/2009 | Travel expenses - gas. |  |  | $16.38  |
|  06/02/2009 | Reimbursable copier and paper charges. |  |  | $17.84  |
|   | Total For Personal Expenses |  |  | $255.20  |
|   |  | Grand Total |  | $3,815.20  |

## Extracción a un esquema empresarial

En producción rara vez alcanza con guardar el Markdown completo. Normalmente se necesita un objeto con campos esperados: proveedor, factura, fecha, total, moneda y líneas de detalle. Para hacerlo explícito, definiremos un esquema con `pydantic` y una función de parsing simple. En un sistema real, esta función podría combinar reglas, LLMs con structured output o validaciones contra catálogos internos.

## Opción 2: DeepSeek-OCR en infraestructura propia

DeepSeek-OCR es un modelo abierto orientado a OCR y conversión de documentos a Markdown. El repositorio oficial muestra dos caminos de inferencia: `vLLM` para despliegues con mayor throughput y `transformers` para pruebas más directas. En una empresa, esta opción resulta interesante cuando se requiere control de datos, despliegue privado o integración con GPU internas.

El siguiente bloque queda desactivado por defecto porque requiere GPU, memoria suficiente y dependencias específicas como `torch`, `flash-attn` o `vLLM`. Se incluye para mostrar el patrón de uso, no como paso obligatorio del notebook.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "deepseek-ai/DeepSeek-OCR"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
).eval().cuda().to(torch.bfloat16)

prompt = "<image>\n<|grounding|>Convert the document to markdown."
output_path = "deepseek_ocr_output"

resultado = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=str(documento_path),
    output_path=output_path,
    base_size=1024,
    image_size=640,
    crop_mode=True,
    save_results=True,
)

print(resultado)

## Arquitectura de referencia

Podemos resumir el flujo empresarial de esta forma:

1. **Ingesta**: recibir PDFs o imágenes desde correo, SFTP, API o almacenamiento interno.
2. **Clasificación ligera**: identificar tipo de documento y prioridad.
3. **OCR avanzado**: procesar con un servicio como Mistral OCR o con un modelo desplegado internamente como DeepSeek-OCR, PaddleOCR o MinerU.
4. **Normalización**: convertir la salida a Markdown/JSON y adjuntar metadatos.
5. **Extracción estructurada**: mapear a esquemas de negocio.
6. **Validación**: aplicar reglas determinísticas, catálogos y umbrales de confianza.
7. **Human-in-the-loop**: derivar a revisión solo los documentos ambiguos.
8. **Persistencia**: guardar documento original, salida OCR, campos extraídos y trazabilidad.

La pregunta práctica sería: ¿qué modelo conviene elegir? Si se prioriza rapidez de adopción, un servicio administrado reduce la complejidad operativa. Si se priorizan datos sensibles, costos a escala o despliegue offline, un modelo abierto desplegado internamente puede ser más adecuado.

## Referencias consultadas

- [Mistral OCR cookbook: structured OCR](https://github.com/mistralai/cookbook/blob/main/mistral/ocr/structured_ocr.ipynb).
- [Mistral OCR cookbook: data extraction via annotations](https://github.com/mistralai/cookbook/blob/main/mistral/ocr/data_extraction.ipynb).
- [DeepSeek-OCR repository](https://github.com/deepseek-ai/DeepSeek-OCR).
- [PaddleOCR repository](https://github.com/PaddlePaddle/PaddleOCR).
- [MinerU repository](https://github.com/opendatalab/MinerU).

## Cierre

Este ejemplo muestra el patrón principal: un modelo OCR avanzado convierte el documento en una representación rica, luego una capa de extracción y validación lo transforma en datos confiables para la empresa. En producción, recuerde evaluar con documentos reales del dominio, medir errores por tipo de campo y conservar evidencia para auditoría.